# Z3-Python 14 — Optimisation multi-objectif (lex, Pareto, somme pondérée)

**Navigation** : [Index](README.md) | [Index SMT](../README.md) | [Précédent](Z3-13-UnsatCores-Python.ipynb)

La série Z3-API s'est concentrée jusqu'ici sur la **satisfiabilité** (SAT/UNSAT, unsat cores, interpolation). Ce carnet ouvre le créneau **optimisation** de Z3 au-delà d'un seul objectif : quand un problème porte **plusieurs objectifs contradictoires** (maximiser l'impact, minimiser le coût), il n'existe pas de « solution optimale » unique. `Optimize()` (Z3 ≥ 4.10) offre trois façons de trancher, que nous comparons sur un même problème jouet :

- **lexicographique** — des priorités hiérarchiques ;
- **somme pondérée** — un unique scalaire `w` ;
- **front de Pareto** — l'ensemble des compromis non dominés.

> Objectif pédagogique : voir que **la même décision, scalarisée de trois manières, donne trois réponses différentes** — et comprendre pourquoi le front de Pareto est l'outil qui rend ce choix visible.

In [1]:
# Imports et verification de l'environnement
# Le package pip s'appelle 'z3-solver' (et non 'z3').
# !pip install z3-solver
import z3
from z3 import Optimize, Int

print("z3-solver", z3.get_version_string())
assert hasattr(z3, "Optimize"), "Optimize() indisponible (Z3 < 4.10 ?)"
print("Optimize() disponible -- OK")

z3-solver 4.16.0
Optimize() disponible -- OK


## Le problème jouet — trois projets, un budget

Trois projets concurrents, chacun avec un **coût** (k€) et un **impact** (score) :

| Projet | Coût (k€) | Impact |
|--------|-----------|--------|
| A | 5 | 7 |
| B | 3 | 4 |
| C | 4 | 6 |

**Budget total : 10 k€.** L'impact d'un portefeuille est la somme additive des impacts, son coût la somme additive des coûts. On peut choisir n'importe quel sous-ensemble de projets tant que le coût reste sous le budget.

La question n'est pas « est-ce faisable ? » (SAT) mais « **quel est le meilleur portefeuille ?** » — et « meilleur » dépend de la façon de comparer impact et coût. Les trois cellules de mesure ci-dessous posent la même question avec trois scalarisations.

In [2]:
# 1. Lexicographique : maximiser l'impact, puis (a impact egal) minimiser le cout.
# Modele commun aux trois techniques : 3 projets, budget 10 kEUR, impact additif.
PROJETS = {
    "A": {"cout": 5, "impact": 7},
    "B": {"cout": 3, "impact": 4},
    "C": {"cout": 4, "impact": 6},
}
BUDGET = 10

def modele(opt):
    """Variables binaires x_A/x_B/x_C + contrainte de budget."""
    x = {n: Int(f"x_{n}") for n in PROJETS}
    for n in PROJETS:
        opt.add(x[n] >= 0, x[n] <= 1)   # binaire
    opt.add(sum(PROJETS[n]["cout"] * x[n] for n in PROJETS) <= BUDGET)
    return x

def impact(x):  return sum(PROJETS[n]["impact"] * x[n] for n in PROJETS)
def cout(x):    return sum(PROJETS[n]["cout"] * x[n] for n in PROJETS)
def portefeuille(x, m):
    return sorted(n for n in PROJETS if m.eval(x[n]) == 1)

opt = Optimize()
x = modele(opt)
opt.maximize(impact(x))   # priorite 1
opt.minimize(cout(x))     # priorite 2 (a impact egal)
opt.check()
m = opt.model()
print("Lexique :", " + ".join(portefeuille(x, m)) or "(vide)",
      "| impact =", m.eval(impact(x)), "| cout =", m.eval(cout(x)), "kEUR")

Lexique : A + C | impact = 13 | cout = 9 kEUR


### Interprétation — lexicographique

Z3 retourne **`A + C`** : impact 13 (7+6), coût 9 k€ (5+4). L'ordre des appels `maximize` puis `minimize` est le contrat : à impact maximal, Z3 minimise le coût parmi les portefeuilles qui atteignent cet impact.

**Vérification manuelle** (la solution est-elle non triviale ?) :
- *A seul* : impact 7, coût 5 — sous-optimal en impact.
- *C seul* : impact 6, coût 4 — sous-optimal en impact aussi.
- *B + C* : impact 10, coût 7 — sous-optimal en impact (10 < 13).
- *A + B* : impact 11, coût 8 — sous-optimal en impact.
- *A + C* : impact 13, coût 9 — **maximal** en impact.
- *A + B + C* : impact 17, **coût 12** (> budget) — infaisable.

À impact = 13, *A + C* est l'unique portefeuille faisable, donc Z3 ne peut pas trancher le second critère. Si la table contenait un autre portefeuille de même impact pour un coût différent, *A + C* gagnerait par coût minimal. Ce comportement reflète exactement la sémantique lexicographique : *priorité 1 ≥ priorité 2*.


In [3]:
# 2. Somme ponderee : un unique scalaire  w * impact - (1-w) * cout.
# Meme modele, meme budget -- seul l'objectif change.
def portefeuille_pondere(w):
    opt = Optimize()
    x = modele(opt)
    opt.maximize(w * impact(x) - (1 - w) * cout(x))
    if opt.check() != z3.sat:
        return "(infeasible)"
    m = opt.model()
    return " + ".join(portefeuille(x, m)) or "(vide)"

print("   w | portefeuille retenu")
for w in (0.0, 0.2, 0.5, 0.8, 1.0):
    print(f"{w:.1f} | {portefeuille_pondere(w)}")


   w | portefeuille retenu
0.0 | (vide)
0.2 | (vide)
0.5 | A + C
0.8 | A + C
1.0 | A + C


### Interprétation — somme pondérée

Le portefeuille retenu **bascule une seule fois** sur la grille mesurée : le vide pour `w` faible, `A + C` dès `w = 0.5`.

| `w` | portefeuille retenu | score `w·impact − (1−w)·cout` |
|------|---------------------|-------------------------------|
| 0.0 | *(vide)* | 0.0 |
| 0.2 | *(vide)* | 0.0 |
| 0.5 | A + C | 2.0 |
| 0.8 | A + C | 8.6 |
| 1.0 | A + C | 13.0 |

**Pourquoi le vide gagne pour `w` faible ?** Le score de chaque portefeuille est une fonction affine de `w` ; le vide a un score constant nul, tandis qu'un portefeuille non vide a un score `w·I − (1−w)·C = (I+C)·w − C`. Ce score ne devient positif que lorsque `w` dépasse le seuil `C / (I + C)`. Pour `B` (coût 3, impact 4) ce seuil vaut `3/7 ≈ 0.43`, pour `C` (coût 4, impact 6) il vaut `4/10 = 0.4`, pour `A + C` (coût 9, impact 13) il vaut `9/22 ≈ 0.41`. Tant que `w` reste sous le seuil le plus bas, le vide — qui coûte 0 — l'emporte. La sortie `(vide)` est donc un optimum **correct**, pas un cas particulier.

**Pourquoi le basculement est brutal.** En comparant les scores des candidats (vide, B, C, A, B+C, A+B, A+C), l'enveloppe supérieure n'est jamais atteinte que par trois points : le vide, puis `C`, puis `A + C`. Les points `B`, `A`, `B + C` et `A + B` — **alors qu'ils sont sur le front de Pareto** — ne sont jamais retenus, à aucun poids. Raison géométrique : un scalaire `w·impact − (1−w)·cout` est une fonction **linéaire** de `(impact, coût)`, donc son maximiseur est toujours un **sommet de l'enveloppe convexe** du front. Or seuls le vide, `C` et `A + C` sont sur cette enveloppe convexe ; les quatre autres sont en dessous.

**Leçon pédagogique** : la somme pondérée **ne parcourt pas le front de Pareto point par point**. Elle ne sélectionne qu'une partie des compromis (ici, 3 sur 7), et le profil `w ↦ portefeuille` est sautillant : entre `w = 0.2` et `w = 0.5` on passe du vide directement à `A + C`, sans jamais visiter `B` ou `C`. C'est la limite classique de la scalarisation linéaire : les portions **non convexes** du front lui sont inaccessibles, quelle que soit la valeur de `w`. C'est précisément l'intérêt de l'énumération exhaustive du front (mesure 3) : elle, elle montre tous les compromis.


In [4]:
# 3. Front de Pareto : tous les portefeuilles non domines (impact max, cout min).
from itertools import combinations

def tous_portefeuilles():
    pts = []
    noms = list(PROJETS)
    for r in range(0, len(noms) + 1):
        for combo in combinations(noms, r):
            c = sum(PROJETS[n]["cout"] for n in combo)
            if c <= BUDGET:
                i = sum(PROJETS[n]["impact"] for n in combo)
                pts.append((i, c, "+".join(combo) or "vide"))
    return pts

def front_pareto(pts):
    front = []
    for (i, c, lab) in pts:
        domine = any(i2 >= i and c2 <= c and (i2 > i or c2 < c)
                     for (i2, c2, _) in pts)
        if not domine:
            front.append((i, c, lab))
    return sorted(front)   # trie par impact

pts = tous_portefeuilles()
front = front_pareto(pts)
print(f"{len(pts)} portefeuilles faisables -> {len(front)} non domines :")
for (i, c, lab) in front:
    print(f"  {lab} : impact={i}  cout={c} kEUR")

7 portefeuilles faisables -> 7 non domines :
  vide : impact=0  cout=0 kEUR
  B : impact=4  cout=3 kEUR
  C : impact=6  cout=4 kEUR
  A : impact=7  cout=5 kEUR
  B+C : impact=10  cout=7 kEUR
  A+B : impact=11  cout=8 kEUR
  A+C : impact=13  cout=9 kEUR


### Interprétation — front de Pareto

Sur les 8 sous-ensembles de `{A, B, C}`, un seul est exclu par le budget : `A + B + C` coûte 12 k€ > 10 k€. Il reste donc **7 portefeuilles faisables**, et **les 7 sont non dominés** :

| portefeuille | impact | coût (k€) |
|---------------|--------|-----------|
| *(vide)* | 0 | 0 |
| B | 4 | 3 |
| C | 6 | 4 |
| A | 7 | 5 |
| B + C | 10 | 7 |
| A + B | 11 | 8 |
| A + C | 13 | 9 |

**Lecture du compromis.** Chaque ligne est un arbitrage distinct, et aucune n'en fait disparaître une autre :

- `*(vide)*` — ne rien faire. Non dominé : aucun portefeuille ne fait mieux sur les deux axes à la fois.
- `B`, `C`, `A` — un seul projet. Chacun est le moins cher pour un impact donné, mais `A` coûte 5 k€ pour 7 points d'impact quand `C` en coûte 4 pour 6.
- `B + C` — le meilleur rapport impact/coût des combinaisons de deux : 10 points pour 7 k€.
- `A + B` — 11 points pour 8 k€.
- `A + C` — le maximum d'impact atteignable sous budget (13), au prix de 9 k€.

**Le vide est bien sur le front.** C'est le point le plus facile à oublier : le portefeuille vide n'est dominé par personne (personne ne fait au moins aussi bien en impact *et* au moins aussi bien en coût), donc il fait partie du front au même titre que les autres. La **dominance** est ici au sens de Pareto strict — un point est dominé s'il existe un autre point au moins aussi bon sur **les deux** axes et strictement meilleur sur l'un des deux. Le front compte donc 7 points, dont **6 portefeuilles non vides** : aucun des 6 points non vides n'est dominé, ce qui est cohérent avec la mesure 2 — ici le front est intégralement non dominé, aucune combinaison ne fait doublon.

**Ce que le front apporte que la somme pondérée ne donne pas.** La mesure 2 ne retenait que trois de ces points (le vide, `C` et `A + C`) : les **quatre** autres (`B`, `A`, `B + C`, `A + B`) sont inaccessibles à un scalaire linéaire, alors qu'ils figurent sur le front. Ici, l'énumération les **expose tous**, y compris des compromis que ni la hiérarchie lexicographique ni le balayage de poids ne présentent jamais — `A + B` (11 pour 8) en particulier. C'est le tableau de négociation que le décideur n'a pas à recalculer lui-même.


## Exercices

Complétez les trois fonctions ci-dessous. Chacune généralise une technique du carnet à **4 projets** (ou 4 poids). Les corps sont des stubs : le carnet s'exécute de bout en bout tant qu'ils ne sont pas complétés.

In [5]:
# EXERCICE 1 -- Lexicographique sur 4 projets
# Ajoutez un 4e projet D = {"cout": 6, "impact": 9} au modele.
# Ecrivez `optimiser_lex(projets, budget)` qui retourne la liste des noms
# du portefeuille choisi par l'optimisation lexicographique
# (maximiser l'impact, puis minimiser le cout a impact egal).
def optimiser_lex(projets, budget):
    # TODO etudiant : construire l'Optimize, les variables binaires,
    # la contrainte de budget, puis maximize(impact) et minimize(cout).
    print("Exercice a completer")
    return None

# A completer : optimiser_lex({"A": {...}, "B": {...}, "C": {...}, "D": {...}}, 12)

In [6]:
# EXERCICE 2 -- Front de Pareto sur 4 projets
# Ecrivez `front_pareto(projets, budget)` qui retourne la liste des
# portefeuilles non domines (tuples (impact, cout, label)), en enumeraut
# tous les sous-ensembles qui restent sous le budget.
def front_pareto(projets, budget):
    # TODO etudiant : enumerez les sous-ensembles faisables, puis
    # eliminez chaque point domine par un autre.
    print("Exercice a completer")
    return None

In [7]:
# EXERCICE 3 -- Somme ponderee sur 4 poids
# Ecrivez `balayage_pondere(projets, budget, poids)` qui, pour chaque w
# de la liste `poids`, retourne le portefeuille retenu par
# maximize(w*impact - (1-w)*cout).
def balayage_pondere(projets, budget, poids):
    # TODO etudiant : pour chaque w, resoudre et collecter le portefeuille.
    print("Exercice a completer")
    return None

## Conclusion — choisir sa scalarisation

| Besoin du décideur | Technique | Ce qu'elle produit |
|--------------------|-----------|--------------------|
| « Quel est LE meilleur, sans ambiguïté ? » (hiérarchie claire) | **Lexicographique** | Un seul portefeuille, fixé par les priorités |
| « J'ai une préférence chiffrée entre impact et coût » | **Somme pondérée** | Un portefeuille pour chaque poids `w` |
| « Montrez-moi TOUS les compromis avant de trancher » | **Front de Pareto** | L'ensemble des portefeuilles non dominés |

**Leçon** : aucune des trois n'est « la bonne » — chacune répond à une question de décision différente. Le front de Pareto est le plus informatif (il ne cache aucun compromis) mais ne tranche pas ; la somme pondérée tranche, mais dépend du choix de `w` ; la version lexicographique est la plus lisible quand une hiérarchie naturelle existe. C'est tout l'intérêt d'`Optimize()` : formaliser ce que le décideur a du mal à exprimer.

### Récapitulatif

Ce carnet a posé un même problème multi-objectif (3 projets, budget 10 k€) sous trois scalarisations `Optimize()` :

1. **Lexicographique** — `maximize` + `minimize` en priorité ;
2. **Somme pondérée** — un scalaire `w·impact − (1−w)·cout` ;
3. **Front de Pareto** — énumération des points non dominés.

Il complète la palette SMT de la série Z3-API : au-delà de la satisfiabilité, Z3 **optimise** — et l'optimisation multi-objectif est exactement le cas où la « meilleure » solution n'est plus unique.